# Credit Risk Prediction — Master Notebook

Notebook này là file thực thi duy nhất dành cho Google Colab. Chọn **Runtime → Run all** để chạy toàn bộ quy trình: setup, EDA, preprocessing, model training, evaluation và kiểm tra deliverables.

Các notebook `01_EDA.ipynb` và `02_Data_Preprocessing.ipynb` vẫn được giữ trong repository như tài liệu/module riêng, nhưng master notebook không gọi hay phụ thuộc vào chúng.

## 0. Environment setup

Clone repository nếu chưa có trong runtime, cài dependencies và chuẩn bị các thư mục đầu ra.

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

REPO_URL = "https://github.com/DuyTuong101/credit-risk-prediction.git"
REPO_DIR_NAME = "credit-risk-prediction"

def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (
            (candidate / "modules" / "data_processing.py").is_file()
            and (candidate / "modules" / "model_training.py").is_file()
            and (candidate / "data").is_dir()
        ):
            return candidate
    return None

PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT is None:
    clone_dir = Path("/content") / REPO_DIR_NAME
    if clone_dir.exists() and find_project_root(clone_dir) is None:
        shutil.rmtree(clone_dir)
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    PROJECT_ROOT = find_project_root(clone_dir)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate project root. Verify REPO_URL and repository access."
    )

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
FEATURES_DIR = PROJECT_ROOT / "features"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"

for folder in [FEATURES_DIR, PROCESSED_DIR, RAW_DIR, MODELS_DIR, REPORTS_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Notebook directory:", Path.cwd())
print("Python:", sys.version)


## 1. Install dependencies

Cài các thư viện được khai báo trong `requirements.txt`. Không cần `nbclient` vì master notebook chạy trực tiếp các bước trong cùng một kernel.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")],
    check=True
)
print("Dependencies are ready.")


## 2. Exploratory Data Analysis (EDA)

Đọc dữ liệu gốc, xem cấu trúc và thống kê, kiểm tra missing values/duplicates, phân phối target, giá trị bất thường, phân phối biến phân loại và tương quan. Các bảng và biểu đồ được hiển thị trực tiếp trong master notebook.

### 1. Import Libraries and Load Data

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display and report figure directory
sns.set_theme(style="whitegrid", palette="muted")
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Read raw data
df = pd.read_csv("../data/raw/credit_risk_dataset.csv")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 2. Descriptive Statistics, Duplicates & Missing Values

In [ ]:
print("=== 1. DATA TYPES INFORMATION ===")
print(df.info())

print(f"\n=== 2. NUMBER OF DUPLICATE ROWS: {df.duplicated().sum()} ===")

print("\n=== 3. MISSING VALUES STATISTICS ===")
missing_df = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
display(missing_df[missing_df['Missing Count'] > 0])

print("\n=== 4. NUMERICAL FEATURES DESCRIPTIVE STATISTICS ===")
display(df.describe().T)

print("\n=== 5. CATEGORICAL FEATURES STATISTICS ===")
display(df.describe(include='object').T)

### 3. Target Variable Distribution (loan_status)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='loan_status', hue='loan_status', palette=['#2b5c8f', '#d95f02'], legend=False)
plt.title("Target Variable Distribution (loan_status: 0 = Non-default, 1 = Default)")
plt.xlabel("Loan Status")
plt.ylabel("Number of Samples")

# Display count and percentage on bars
total = len(df)
for p in ax.patches:
    count = int(p.get_height())
    pct = 100 * count / total
    ax.annotate(f'{count}\n({pct:.1f}%)', (p.get_x() + p.get_width() / 2., count),
                ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

plt.ylim(0, total * 0.9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_target_distribution.png", dpi=300)
plt.show()

### 4. Outliers and Invalid Values Detection

In [ ]:
import sys

sys.path.append(os.path.abspath(".."))
from modules.data_processing import audit_constraints, clean_raw_data

num_cols = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length'
]

print("=== 4.1 Constraint violations in RAW data ===")
display(audit_constraints(df).to_frame("rows_violating"))

def iqr_summary(data: pd.DataFrame, cols: list) -> pd.DataFrame:
    rows = []
    for c in cols:
        s = data[c].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((s < lo) | (s > hi)).sum())
        rows.append({
            "feature": c, "min": s.min(), "max": s.max(),
            "lower_fence": round(lo, 2), "upper_fence": round(hi, 2),
            "n_outliers": n_out, "outliers_%": round(100 * n_out / len(s), 2),
            "skewness": round(s.skew(), 2),
        })
    return pd.DataFrame(rows).set_index("feature")

print("\n=== 4.2 IQR outlier summary (RAW data) ===")
display(iqr_summary(df, num_cols))

# Apply the cleaning rules used in the pipeline, then re-check
df_clean = clean_raw_data(df, verbose=True)
print("\n=== 4.2 IQR outlier summary (CLEANED data) ===")
display(iqr_summary(df_clean, num_cols))

def plot_boxplots(data: pd.DataFrame, title: str, path: str):
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()
    for i, col in enumerate(num_cols):
        sns.boxplot(data=data, y=col, ax=axes[i], color='#4c72b0')
        axes[i].set_title(f"{col}")
        if col == 'person_income':          # heavy right skew -> log scale
            axes[i].set_yscale('log')
            axes[i].set_ylabel('person_income (log scale)')
    axes[-1].axis('off')
    fig.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()


plot_boxplots(df, "Boxplots - RAW data (before cleaning)",
              f"{FIG_DIR}/02a_boxplots_raw.png")
plot_boxplots(df_clean, "Boxplots - CLEANED data (after removing invalid rows)",
              f"{FIG_DIR}/02b_boxplots_cleaned.png")


print("Records with person_age > 100:")
display(df[df['person_age'] > 100])

print("Records with person_emp_length > 60 or > age - 14:")
display(df[df['person_emp_length'] > df['person_age'] - 14])


### 5. Categorical Distributions by loan_status

In [ ]:
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = sorted(df[col].unique()) if col == 'loan_grade' else df[col].value_counts().index
    sns.countplot(data=df, x=col, hue='loan_status', order=order, ax=axes[i], palette='Set2')
    axes[i].set_title(f"{col} by Loan Status")
    axes[i].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_categorical_distributions.png", dpi=300)
plt.show()

### 6. Correlation Heatmap

In [ ]:
plt.figure(figsize=(9, 7))
corr_matrix = df[num_cols + ['loan_status']].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Matrix of Numerical Features and loan_status")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_correlation_heatmap.png", dpi=300)
plt.show()

## 3. Data Preprocessing

Làm sạch dữ liệu, chia train/test và tạo 8 cấu hình feature. Các preprocessing transformers chỉ được fit trên tập train để tránh data leakage.

### 1. Import Libraries and Load Data

In [ ]:
import os
import sys
import urllib.request
import numpy as np
import pandas as pd
 
sys.path.append(os.path.abspath(".."))
from modules.data_processing import (
    clean_raw_data, split_data, build_preprocessing_pipeline,
    get_feature_names, save_features,
)
 
RAW_PATH = "../data/raw/credit_risk_dataset.csv"
RAW_URL = "https://raw.githubusercontent.com/DuyTuong101/credit-risk-prediction/main/data/raw/credit_risk_dataset.csv"     
FEATURES_DIR = "../features"
PROCESSED_DIR = "../data/processed"
for d in (os.path.dirname(RAW_PATH), FEATURES_DIR, PROCESSED_DIR):
    os.makedirs(d, exist_ok=True)
 
if not os.path.exists(RAW_PATH):                
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)
 
df_raw = pd.read_csv(RAW_PATH)
print("Initial shape:", df_raw.shape)

### 2. Data Cleaning and Train/Test Split (Preventing Data Leakage)

In [ ]:
df_clean = clean_raw_data(df_raw)              
df_clean.to_csv(f"{PROCESSED_DIR}/credit_risk_cleaned.csv", index=False)
 

X_train, X_test, y_train, y_test = split_data(df_clean, target_col="loan_status")
 
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Default rate Train: {y_train.mean():.4f} | Test: {y_test.mean():.4f}")


### 3. Run Baseline Configuration and Save Features

In [ ]:
num_cols = ['person_age', 'person_income', 'person_emp_length',
            'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
ordinal_cols = ['loan_grade']
nominal_cols = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']
 
# KNNImputer is excluded from this full-data run because its pairwise distance
# computation can require excessive RAM on datasets with 30k+ rows.
configs = {
    "baseline":         dict(impute_strategy="median", scaler_type="standard"),
    "minmax_0_1":       dict(impute_strategy="median", scaler_type="minmax", feature_range=(0, 1)),
    "minmax_m1_1":      dict(impute_strategy="median", scaler_type="minmax", feature_range=(-1, 1)),
    "robust_logincome": dict(impute_strategy="median", scaler_type="robust", log_cols=["person_income"]),
    "median_indicator":  dict(impute_strategy="median", scaler_type="standard", add_missing_indicator=True),
    "standard_pca90":   dict(impute_strategy="median", scaler_type="standard", pca_variance=0.90),
    "standard_pca95":   dict(impute_strategy="median", scaler_type="standard", pca_variance=0.95),
    "minmax_pca95":     dict(impute_strategy="median", scaler_type="minmax", feature_range=(0, 1), pca_variance=0.95),
}
 
summary_rows = []
for name, cfg in configs.items():
    pipe = build_preprocessing_pipeline(num_cols, ordinal_cols, nominal_cols, **cfg)
    X_tr = pipe.fit_transform(X_train)           # fit CHỈ trên train
    X_te = pipe.transform(X_test)                # test chỉ transform
 
    assert not np.isnan(X_tr).any() and not np.isnan(X_te).any()
    assert X_tr.shape[1] == X_te.shape[1] == len(get_feature_names(pipe))
 
    save_features(name, pipe, X_tr, X_te, y_train, y_test, cfg,
                  out_dir=FEATURES_DIR, save_pipeline=(name == "baseline"))
 
    summary_rows.append({
        "Config": name,
        "Imputer": cfg.get("impute_strategy"),
        "Scaler": cfg.get("scaler_type"),
        "Log cols": ",".join(cfg.get("log_cols", [])) or "-",
        "Missing indicator": cfg.get("add_missing_indicator", False),
        "PCA variance": cfg.get("pca_variance") or "-",
        "n_features": X_tr.shape[1],
    })
 
print("Baseline features:", get_feature_names(
    build_preprocessing_pipeline(num_cols, ordinal_cols, nominal_cols).fit(X_train)))
display(pd.DataFrame(summary_rows))

### 4. Test Configurable Pipeline Options

In [ ]:
import glob, json
print(sorted(os.path.basename(p) for p in glob.glob(f"{FEATURES_DIR}/*")))
y_tr, y_te = np.load(f"{FEATURES_DIR}/y_train.npy"), np.load(f"{FEATURES_DIR}/y_test.npy")
for name in configs:
    a = np.load(f"{FEATURES_DIR}/X_train_{name}.npy"); b = np.load(f"{FEATURES_DIR}/X_test_{name}.npy")
    assert a.shape[0] == len(y_tr) and b.shape[0] == len(y_te) and a.shape[1] == b.shape[1]
print("OK: X/y align", len(configs), "configs")

In [ ]:
# ------------------------------------------------------------
# Feature artifact metadata and reproducibility manifest
# ------------------------------------------------------------
import json
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

FEATURES_PATH = Path(FEATURES_DIR)
FEATURES_PATH.mkdir(parents=True, exist_ok=True)

# Recreate the row indices using the exact same stratified split settings
# as split_data(). This records which original cleaned-data rows belong
# to train and test without changing the feature matrices.
all_indices = np.arange(len(df_clean))
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.2,
    random_state=42,
    stratify=df_clean["loan_status"].values,
)
np.save(FEATURES_PATH / "train_indices.npy", train_indices)
np.save(FEATURES_PATH / "test_indices.npy", test_indices)

y_train_saved = np.load(FEATURES_PATH / "y_train.npy")
y_test_saved = np.load(FEATURES_PATH / "y_test.npy")

manifest = {
    "dataset": "credit_risk_dataset.csv",
    "target": "loan_status",
    "split": {
        "method": "stratified train_test_split",
        "test_size": 0.2,
        "random_state": 42,
        "train_rows": int(len(train_indices)),
        "test_rows": int(len(test_indices)),
        "train_indices_file": "train_indices.npy",
        "test_indices_file": "test_indices.npy",
        "y_train_file": "y_train.npy",
        "y_test_file": "y_test.npy",
    },
    "feature_configurations": {},
    "notes": [
        "Each preprocessing pipeline is fitted only on X_train and then applied to X_test.",
        "Feature arrays are exported artifacts for inspection and reuse.",
        "Model evaluation independently fits preprocessing inside each CV fold to prevent data leakage.",
    ],
}

for config_name, config in configs.items():
    Xtr_path = FEATURES_PATH / f"X_train_{config_name}.npy"
    Xte_path = FEATURES_PATH / f"X_test_{config_name}.npy"
    names_path = FEATURES_PATH / f"feature_names_{config_name}.json"
    config_path = FEATURES_PATH / f"config_{config_name}.json"

    Xtr = np.load(Xtr_path)
    Xte = np.load(Xte_path)
    feature_names = json.loads(names_path.read_text(encoding="utf-8"))
    config_metadata = json.loads(config_path.read_text(encoding="utf-8"))

    assert Xtr.shape[0] == len(y_train_saved), f"{config_name}: train row mismatch"
    assert Xte.shape[0] == len(y_test_saved), f"{config_name}: test row mismatch"
    assert Xtr.shape[1] == Xte.shape[1] == len(feature_names), (
        f"{config_name}: feature dimension/name mismatch"
    )
    assert np.isfinite(Xtr).all() and np.isfinite(Xte).all(), (
        f"{config_name}: features contain NaN or infinity"
    )

    manifest["feature_configurations"][config_name] = {
        "X_train_file": Xtr_path.name,
        "X_test_file": Xte_path.name,
        "feature_names_file": names_path.name,
        "config_file": config_path.name,
        "train_shape": list(Xtr.shape),
        "test_shape": list(Xte.shape),
        "number_of_features": int(Xtr.shape[1]),
        "feature_names": feature_names,
        "preprocessing_config": config_metadata,
        "dtype": str(Xtr.dtype),
        "finite_values": True,
    }

manifest["target_distribution"] = {
    "train": {
        str(label): int(count)
        for label, count in zip(*np.unique(y_train_saved, return_counts=True))
    },
    "test": {
        str(label): int(count)
        for label, count in zip(*np.unique(y_test_saved, return_counts=True))
    },
}

manifest_path = FEATURES_PATH / "feature_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("Feature manifest saved:", manifest_path)
print("Train/test row indices saved.")
print("All feature configurations passed shape, label-alignment and finite-value checks.")
display(pd.DataFrame([
    {
        "configuration": name,
        "train_rows": meta["train_shape"][0],
        "test_rows": meta["test_shape"][0],
        "n_features": meta["number_of_features"],
        "dtype": meta["dtype"],
        "finite": meta["finite_values"],
    }
    for name, meta in manifest["feature_configurations"].items()
]))


### Feature artifact và chống data leakage

Notebook xuất các ma trận `X_train`/`X_test`, nhãn `y_train`/`y_test`, tên đặc trưng, cấu hình preprocessing và chỉ số dòng train/test vào `features/`. Các file này là deliverables để kiểm tra, tái sử dụng và tái lập phép biến đổi.

Trong phần đánh giá mô hình, `model_training.py` vẫn tự fit preprocessing bên trong từng fold Cross Validation thay vì dùng trực tiếp ma trận đã fit một lần ở bước trên. Đây là chủ ý: mỗi fold phải tự học imputer/scaler/encoder/PCA chỉ từ phần training của fold đó, tránh rò rỉ thông tin validation. Hai phần dùng cùng loại biến đổi và cùng seed chia dữ liệu, nhưng phục vụ hai mục đích khác nhau.


In [ ]:
# Restore Path objects after the preprocessing section uses relative string paths.
FEATURES_DIR = PROJECT_ROOT / "features"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"
print("Output directories:", FEATURES_DIR, PROCESSED_DIR, MODELS_DIR, REPORTS_DIR)


## 4. Model Training and Evaluation

Sử dụng hàm `train_and_evaluate()` hiện có trong `modules/model_training.py`. So sánh Logistic Regression, SVM và Random Forest bằng Stratified 3-fold Cross Validation, sau đó đánh giá mô hình tốt nhất trên held-out test set.

## 4. Model Training, Experiments & Evaluation

Thực nghiệm được chia thành hai phần. **Experiment A** so sánh bốn cấu hình preprocessing (StandardScaler, MinMaxScaler, PCA 90% và PCA 95%) trên cùng ba mô hình. **Experiment B** thay đổi một số hyperparameter của từng mô hình trên cấu hình preprocessing được chọn từ kết quả Cross-Validation.

Các lần chạy dùng cùng `random_state=42`, tỷ lệ test 20% và Stratified 3-Fold Cross Validation. SVM dùng kernel tuyến tính trong thực nghiệm chính để giảm chi phí tính toán trên toàn bộ dataset. Việc chọn cấu hình dựa trên mean CV F1; tập test được giữ riêng để đánh giá mô hình cuối cùng.

Kết quả được lưu tại `reports/preprocessing_experiments.csv`, `reports/hyperparameter_experiments.csv` và các thư mục con trong `models/experiments/`, `models/hyperparameters/`.


In [ ]:
import json
import shutil
import pandas as pd
from pathlib import Path
from IPython.display import display
from modules.model_training import train_and_evaluate

CLEANED_DATA_PATH = PROCESSED_DIR / "credit_risk_cleaned.csv"
if not CLEANED_DATA_PATH.is_file():
    raise FileNotFoundError(
        "The cleaned dataset was not produced by the preprocessing step."
    )

df_clean = pd.read_csv(CLEANED_DATA_PATH)
print("Cleaned dataset shape:", df_clean.shape)
print("Target distribution:")
display(df_clean["loan_status"].value_counts(normalize=True).rename("proportion"))

# ------------------------------------------------------------
# EXPERIMENT A: Compare preprocessing configurations.
# Every run uses the same split seed and the same 3-fold CV.
# ------------------------------------------------------------
preprocessing_experiments = [
    {
        "experiment": "standard_no_pca",
        "scaler": "standard",
        "pca_variance": None,
    },
    {
        "experiment": "minmax_no_pca",
        "scaler": "minmax",
        "pca_variance": None,
    },
    {
        "experiment": "standard_pca90",
        "scaler": "standard",
        "pca_variance": 0.90,
    },
    {
        "experiment": "standard_pca95",
        "scaler": "standard",
        "pca_variance": 0.95,
    },
]

# Linear SVM is used for the main comparison to keep the full-data
# experiment computationally practical. RBF SVM can be evaluated separately.
base_model_params = {
    "logistic_regression": {"C": 1.0},
    "svm": {"C": 1.0, "kernel": "linear"},
    "random_forest": {"n_estimators": 100, "max_depth": None},
}
model_names = ("logistic_regression", "svm", "random_forest")

experiment_rows = []
experiment_artifacts = {}

for config in preprocessing_experiments:
    exp_name = config["experiment"]
    print(f"\n{'=' * 70}\nRunning preprocessing experiment: {exp_name}\n{'=' * 70}")

    exp_models_dir = MODELS_DIR / "experiments" / exp_name
    exp_reports_dir = REPORTS_DIR / "experiments" / exp_name

    result = train_and_evaluate(
        df=df_clean,
        target_col="loan_status",
        test_size=0.2,
        cv_folds=3,
        random_state=42,
        scaler=config["scaler"],
        pca_variance=config["pca_variance"],
        numeric_imputation="median",
        categorical_imputation="most_frequent",
        model_names=model_names,
        model_params=base_model_params,
        models_dir=exp_models_dir,
        reports_dir=exp_reports_dir,
    )

    comparison = result["comparison"].copy()
    comparison["experiment"] = exp_name
    comparison["scaler"] = config["scaler"]
    comparison["pca_variance"] = config["pca_variance"]
    experiment_rows.extend(comparison.to_dict(orient="records"))

    # The module saves the best model for this preprocessing configuration.
    experiment_artifacts[exp_name] = {
        "model_path": Path(result["model_path"]),
        "reports_dir": exp_reports_dir,
        "config": config,
    }

preprocessing_results = pd.DataFrame(experiment_rows)
preprocessing_results = preprocessing_results[
    [
        "experiment", "scaler", "pca_variance", "model",
        "cv_f1_mean", "cv_f1_std", "cv_roc_auc_mean",
    ]
].sort_values("cv_f1_mean", ascending=False)

print("\nExperiment A — preprocessing comparison:")
display(preprocessing_results)

# Select the preprocessing setup associated with the strongest CV F1.
best_preprocessing_experiment = (
    preprocessing_results.sort_values("cv_f1_mean", ascending=False)
    .iloc[0]["experiment"]
)
best_preprocessing_config = next(
    item for item in preprocessing_experiments
    if item["experiment"] == best_preprocessing_experiment
)
print("Selected preprocessing configuration:", best_preprocessing_config)

# ------------------------------------------------------------
# EXPERIMENT B: Hyperparameter comparison using the selected
# preprocessing configuration. Each run evaluates one model.
# ------------------------------------------------------------
hyperparameter_experiments = [
    ("logistic_regression", "lr_C_0.5", {"C": 0.5}),
    ("logistic_regression", "lr_C_1.0", {"C": 1.0}),
    ("svm", "svm_linear_C_0.5", {"C": 0.5, "kernel": "linear"}),
    ("svm", "svm_linear_C_1.0", {"C": 1.0, "kernel": "linear"}),
    ("random_forest", "rf_100_depth_none", {"n_estimators": 100, "max_depth": None}),
    ("random_forest", "rf_150_depth_12", {"n_estimators": 150, "max_depth": 12}),
]

hyperparameter_rows = []
hyperparameter_artifacts = {}

for model_name, exp_name, params in hyperparameter_experiments:
    print(f"\nRunning hyperparameter experiment: {exp_name}")

    exp_models_dir = MODELS_DIR / "hyperparameters" / exp_name
    exp_reports_dir = REPORTS_DIR / "hyperparameters" / exp_name

    result = train_and_evaluate(
        df=df_clean,
        target_col="loan_status",
        test_size=0.2,
        cv_folds=3,
        random_state=42,
        scaler=best_preprocessing_config["scaler"],
        pca_variance=best_preprocessing_config["pca_variance"],
        numeric_imputation="median",
        categorical_imputation="most_frequent",
        model_names=(model_name,),
        model_params={model_name: params},
        models_dir=exp_models_dir,
        reports_dir=exp_reports_dir,
    )

    row = result["comparison"].iloc[0].to_dict()
    row.update({
        "experiment": exp_name,
        "model": model_name,
        "parameters": json.dumps(params, sort_keys=True),
        "scaler": best_preprocessing_config["scaler"],
        "pca_variance": best_preprocessing_config["pca_variance"],
        "test_accuracy": result["test_metrics"]["accuracy"],
        "test_precision": result["test_metrics"]["precision"],
        "test_recall": result["test_metrics"]["recall"],
        "test_f1": result["test_metrics"]["f1"],
        "test_roc_auc": result["test_metrics"]["roc_auc"],
    })
    hyperparameter_rows.append(row)

    hyperparameter_artifacts[exp_name] = {
        "model_path": Path(result["model_path"]),
        "reports_dir": exp_reports_dir,
        "test_metrics": result["test_metrics"],
    }

hyperparameter_results = pd.DataFrame(hyperparameter_rows).sort_values(
    "cv_f1_mean", ascending=False
)

print("\nExperiment B — hyperparameter comparison:")
display(hyperparameter_results)

# ------------------------------------------------------------
# Save consolidated experiment results and promote the best
# CV-selected model/report artifacts to the standard locations.
# ------------------------------------------------------------
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

preprocessing_results.to_csv(
    REPORTS_DIR / "preprocessing_experiments.csv", index=False
)
hyperparameter_results.to_csv(
    REPORTS_DIR / "hyperparameter_experiments.csv", index=False
)

best_hyperparameter_row = hyperparameter_results.iloc[0]
best_experiment_name = best_hyperparameter_row["experiment"]
best_artifact = hyperparameter_artifacts[best_experiment_name]

shutil.copy2(best_artifact["model_path"], MODELS_DIR / "best_model.joblib")

for filename in (
    "test_metrics.json",
    "classification_report.json",
    "confusion_matrix.csv",
):
    source_file = best_artifact["reports_dir"] / filename
    if source_file.is_file():
        shutil.copy2(source_file, REPORTS_DIR / filename)

final_test_metrics = best_artifact["test_metrics"]
with (REPORTS_DIR / "test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(final_test_metrics, f, indent=2, default=str)

print("\nFinal selected model:")
print("Experiment:", best_experiment_name)
print("Model:", best_hyperparameter_row["model"])
print("CV F1:", round(float(best_hyperparameter_row["cv_f1_mean"]), 4))
print("CV ROC-AUC:", round(float(best_hyperparameter_row["cv_roc_auc_mean"]), 4))
print("Held-out test metrics:")
display(pd.DataFrame([final_test_metrics]).T.rename(columns={0: "value"}))
print("Final model saved to:", MODELS_DIR / "best_model.joblib")
print("Experiment tables saved to:", REPORTS_DIR)


## 5. Verify Generated Deliverables

Kiểm tra các file mô hình, báo cáo đánh giá và feature arrays đã được tạo đầy đủ.

In [ ]:
expected_outputs = [
    MODELS_DIR / "best_model.joblib",
    REPORTS_DIR / "model_comparison.csv",
    REPORTS_DIR / "test_metrics.json",
    REPORTS_DIR / "classification_report.json",
    REPORTS_DIR / "confusion_matrix.csv",
    REPORTS_DIR / "preprocessing_experiments.csv",
    REPORTS_DIR / "hyperparameter_experiments.csv",
    FEATURES_DIR / "X_train_baseline.npy",
    FEATURES_DIR / "X_test_baseline.npy",
    FEATURES_DIR / "y_train.npy",
    FEATURES_DIR / "y_test.npy",
    FEATURES_DIR / "feature_manifest.json",
    FEATURES_DIR / "train_indices.npy",
    FEATURES_DIR / "test_indices.npy",
    FEATURES_DIR / "feature_names_baseline.json",
    FEATURES_DIR / "config_baseline.json",
    FEATURES_DIR / "preprocessor_baseline.joblib",
]

verification = pd.DataFrame({
    "file": [str(p.relative_to(PROJECT_ROOT)) for p in expected_outputs],
    "exists": [p.is_file() for p in expected_outputs],
})
display(verification)

missing = verification.loc[~verification["exists"], "file"].tolist()
if missing:
    raise FileNotFoundError("Missing expected output(s): " + ", ".join(missing))

print("\nAll required outputs in this notebook's verification list exist.")


## 6. Pipeline Completed

Sau khi tất cả cell chạy thành công, các artifact được lưu trong repository tại `reports/`, `features/` và `models/`.

In [ ]:
print("Master notebook pipeline completed successfully.")
print("EDA figures:", FIG_DIR)
print("Cleaned data:", CLEANED_DATA_PATH)
print("Model comparison:", REPORTS_DIR / "model_comparison.csv")
print("Test metrics:", REPORTS_DIR / "test_metrics.json")
print("Best model:", MODELS_DIR / "best_model.joblib")
